In [ ]:
import sys 
import os
sys.path.append('../')
from services.supabase_client import supabase
user_id = "51396b9a-279e-4ccd-981e-a0453c1a5688"
files = supabase.storage.from_("resumes").list(user_id)
files
resume_bytes = supabase.storage.from_("resumes").download(
    f"{user_id}/resume.pdf"
)
resume_bytes
import tempfile
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader

with tempfile.NamedTemporaryFile(
    suffix=".pdf",
    delete=False
) as temp_file:

    temp_file.write(resume_bytes)

    temp_path = temp_file.name
loader = PyMuPDFLoader(temp_path)

documents = loader.load()
os.remove(temp_path)
documents
from langchain_text_splitters import RecursiveCharacterTextSplitter



def split_documents(documents,chunk_size=1000,chunk_overlap=200):

    """

    Split documents into smaller chunks for better RAG performance.

    

    Parameters:

    - chunk_size: Maximum characters per chunk (adjust based on your LLM)

    - chunk_overlap: Characters to overlap between chunks (preserves context)

    """

    text_splitter = RecursiveCharacterTextSplitter(

        chunk_size=chunk_size, # Each chunk: ~1000 characters

        chunk_overlap=chunk_overlap, # 200 chars overlap for context

        length_function=len, # How to measure length

        separators=["\n\n", "\n", " ", ""] # Split hierarchy

    )

    # Actually split the documents

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    

    # Show what a chunk looks like

    if split_docs:

        print(f"\nExample chunk:")

        print(f"Content: {split_docs[0].page_content[:200]}...")

        print(f"Metadata: {split_docs[0].metadata}")

    

    return split_docs



chunks=split_documents(documents)

chunks

from services.embedding_manager import embedding_manager
from services.vector_store import vector_store

texts = [
    doc.page_content
    for doc in chunks
]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(
    chunks,
    embeddings,
    user_id
)

import sys
sys.path.append("../")
from services.llm import llm

response = llm.invoke("Say hello in one sentence.")

print(response.content)

from services.embedding_manager import embedding_manager

embeddings = embedding_manager.generate_embeddings([
    "Hello world"
])

print(embeddings.shape)

from services.vector_store import vector_store

print(vector_store.collection.count())

from services.retriever import retriever

user_id = "51396b9a-279e-4ccd-981e-a0453c1a5688"

results = retriever.retrieve(
    "What skills does this person have?",
    user_id
)

print(results)

from services.rag_service import rag_service

user_id = "51396b9a-279e-4ccd-981e-a0453c1a5688"

answer = rag_service.answer_question(
    user_id=user_id,
    question="What skills does he have"
)

print(answer)

texts = [
    doc.page_content 
    for doc in chunks
]



Split 1 documents into 3 chunks

Example chunk:
Content: Arnav Kekre[4pt] Indore, Madhya Pradesh, India GitHub | LinkedIn
Education
Institute of Engineering and Technology (IET), DAVV
2024 – Present B.Tech. in Computer
Science and Business Systems (CSBS) CG...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-10T09:20:31+00:00', 'source': 'C:\\Users\\Arnav\\AppData\\Local\\Temp\\tmpdrzvjivx.pdf', 'file_path': 'C:\\Users\\Arnav\\AppData\\Local\\Temp\\tmpdrzvjivx.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-10T09:20:31+00:00', 'trapped': '', 'modDate': 'D:20260610092031Z', 'creationDate': 'D:20260610092031Z', 'page': 0}
Generating Embeddings for 3 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.41it/s]

Generate Embeddings with shape: (3, 384)
Adding 3 documents to vector store
successful added 3 documents to vector store
Total documents in collection: 3


Hello, how can I assist you today?
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.01it/s]


...

Batches: 100%|██████████| 1/1 [00:00<00:00, 124.82it/s]


Generate Embeddings with shape: (1, 384)
[{'content': '• Developed a full-stack quiz platform using Node.js, FastAPI, Python, HTML, CSS, and JavaScript.\n• Integrated Supabase for database management and user data storage.\n• Built an automated feedback system using a RoBERTa model through Hugging Face APIs.\n• Designed quiz generation, evaluation, and performance tracking workflows.\n• Implemented backend APIs and database interactions for scalable user management.\nCrowd Counting using YOLO\n• Developed a computer vision application for crowd detection and counting using YOLO.\n• Processed image/video streams to estimate crowd density in real time.\nDjango Shop Order Placement System\n• Built an e-commerce order placement workflow using Django.\n• Implemented product listing, order management, and database integration.\nRAG-Based Document Summarization Pipeline\n• Developed a Retrieval-Augmented Generation pipeline for document summarization.\n• Implemented document retrieval, contex

Batches: 100%|██████████| 1/1 [00:00<00:00, 68.98it/s]

Generate Embeddings with shape: (1, 384)


Yes, the candidate has several projects listed on their resume. Here are the projects mentioned:

1. AI-Powered Full Stack Quiz Application
   - Developed a full-stack quiz platform using Node.js, FastAPI, Python, HTML, CSS, and JavaScript.
   - Integrated Supabase for database management and user data storage.
2. Crowd Counting using YOLO
   - Developed a computer vision application for crowd detection and counting using YOLO.
   - Processed image/video streams to estimate crowd density in real time.
3. Django Shop Order Placement System
   - Built an e-commerce order placement workflow using Django.
   - Implemented product listing, order management, and database integration.
4. RAG-Based Document Summarization Pipeline
   - Developed a Retrieval-Augmented Generation pipeline for document summarization.
   - Implemented document retrieval, context generation, and summary creation workflows.


In [13]:
import sys
sys.path.append("../")
from services.llm import llm

response = llm.invoke("Say hello in one sentence.")

print(response.content)

Hello, how can I assist you today?


In [14]:
from services.embedding_manager import embedding_manager

embeddings = embedding_manager.generate_embeddings([
    "Hello world"
])

print(embeddings.shape)

Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.91it/s]

Generate Embeddings with shape: (1, 384)
(1, 384)


In [15]:
from services.vector_store import vector_store

print(vector_store.collection.count())

0


In [16]:
from services.retriever import retriever

user_id = "51396b9a-279e-4ccd-981e-a0453c1a5688"

results = retriever.retrieve(
    "What skills does this person have?",
    user_id
)

print(results)

Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 80.98it/s]

Generate Embeddings with shape: (1, 384)
[]


In [19]:
from services.rag_service import rag_service

user_id = "51396b9a-279e-4ccd-981e-a0453c1a5688"

answer = rag_service.answer_question(
    user_id=user_id,
    question="What programming languages does this candidate know?"
)

print(answer)

Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.79it/s]

Generate Embeddings with shape: (1, 384)


I couldn't find any information about the candidate's programming languages in the provided resume context.


In [13]:
vector_store.collection.count()

1

In [11]:
sys.path.append('../')
from services.supabase_client import supabase

In [12]:
user_id = "51396b9a-279e-4ccd-981e-a0453c1a5688"

In [13]:
files = supabase.storage.from_("resumes").list(user_id)
files

[{'name': 'resume.pdf',
  'id': '6bec7af1-e0e0-4968-8f14-80548b51117f',
  'updated_at': '2026-07-21T05:43:23.711Z',
  'created_at': '2026-07-21T05:43:23.711Z',
  'last_accessed_at': '2026-07-21T05:43:23.711Z',
  'metadata': {'eTag': '"480ed3894bd3409a0bf879c0c41ed49e"',
   'size': 319801,
   'mimetype': 'application/pdf',
   'cacheControl': 'no-cache',
   'lastModified': '2026-07-21T05:43:24.000Z',
   'contentLength': 319801,
   'httpStatusCode': 200}}]

In [14]:
resume_bytes = supabase.storage.from_("resumes").download(
    f"{user_id}/resume.pdf"
)
resume_bytes

b'%PDF-1.5\n%\xe2\xe3\xcf\xd3\n9 0 obj\n<<\n/Ordering (Identity)\n/Registry (Adobe)\n/Supplement 0\n>>\nendobj\n11 0 obj\n<<\n/Filter /FlateDecode\n/Length 148310\n/Length1 341484\n>>\nstream\nx\x9c\xec\x9d\t\\\xd4U\xd7\xf8\xcf\xfd\xfdf_`\x86\x1d\x86a\x06\x86\x19\xf6a\x15\x05\x15F\x04\x04q\x01a\x0c\xb4\x12\x14\x14+\xd3T4M\x13\xb3\xcc\xc8\xb2\xbd\xb4\xcd\xf6\xc5\xcaa\xd4\xc2\xb4\xb2\xb2\xe5\xc9l\xb5}\xb3\xa7\x9eV)+m\xd1\x9c\xf9\x9f\xfb;\x03\xa2i\xefS\xff\xde\xa7\xf7\xfd\xbcs\xe0\xfc\xbe\xf7\x9e{\xee\xbe\xfc\xee\x08\x110\x00\x88\xc2\x87\x0c\x9a\xcb\xeb\xabG\xa5\xac\xad\x8e\x06yL*\x80ygEYy\x83\xf0\x81;\x03X\xf2\xdb\x00\xca\xad\x15ecF\xda>}\xa4\x10\x98\xf5g\x00q\xef\xa8\xf2\x8a\xca\x89?\x9ea\x05!\xb1\x13@a\x18U;\xbe~F\xe3)/\x83\x90r\x05\x08\xfb\xaf\x1bU\xef.;\xb8r\xc6z`\x99\x07\x00\x0et\x8d\xaf\xcf\xce\xbb\xf1\xdb)\xdf\x01\xb0/\xb0\xd6\xe6i\xb3Z\xe6\\\xb4br!\xc0x\xe0\xe5M[0\xdfZ\xfah\xe3>\x80\xb9X\x9fb\xd5\xf493f\xdd\xf6U\xd9\x03\x00\x13\xf6\x02\xa8\xc3g\xb4\xcc\x9b\x031`\xc3\xfay~\xc3\x8c

In [17]:
import tempfile
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader

with tempfile.NamedTemporaryFile(
    suffix=".pdf",
    delete=False
) as temp_file:

    temp_file.write(resume_bytes)

    temp_path = temp_file.name
loader = PyMuPDFLoader(temp_path)

documents = loader.load()
documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-04-30T06:51:50+00:00', 'source': 'C:\\Users\\Arnav\\AppData\\Local\\Temp\\tmp40l7u24a.pdf', 'file_path': 'C:\\Users\\Arnav\\AppData\\Local\\Temp\\tmp40l7u24a.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'sapnakek.18@gmail.com', 'subject': '', 'keywords': '', 'moddate': '2026-04-30T06:51:51+00:00', 'trapped': '', 'modDate': 'D:20260430065151Z', 'creationDate': "D:20260430065150+00'00'", 'page': 0}, page_content='Operating System – Introduction Paragraph \nAn Operating System (OS) is system software that acts as an interface between the user and \ncomputer hardware. It manages all the resources of a computer, including CPU, memory, storage, \nand input/output devices, and ensures efficient execution of programs. The operating system also \nprovides a user-friendly environment for running applications and handling multiple tasks \nsimultaneously through proce

In [ ]:
results = retriever.retrieve(
    "What skills does this person have?",
    user_id
)
for r in results:
    print(r["content"])
    print("----------------")

In [12]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader
dir_loader= DirectoryLoader(
    "../data/pdf", 
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
pdf_documents= dir_loader.load()
pdf_documents

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks=split_documents(pdf_documents)
chunks

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Any, List, Tuple, Dict
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str ="all-MiniLM-L6-v2"):
        self.model_name= model_name
        self.model= None
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str])->np.ndarray:
        if not self.model: 
            raise ValueError("Model not laoded")
        print(f"Generating Embeddings for {len(texts)} texts...")
        embeddings= self.model.encode(texts, show_progress_bar=True)
        print(f"Generate Embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager

In [ ]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str= "../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory= persist_directory
        self.client= None
        self.collection= None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection= self.client.get_or_create_collection(
                name=self.collection_name,
                metadata= {'description':'PDF document embeddings for RAG'}
            )
            print(f"Vector stored initialized. Collection: {self.collection_name}")
            print(f"Existing document int colelction: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents : List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number embeddings")
        print(f"Adding {len(documents)} documents to vector store")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id= f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata= dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings= embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successful added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store {e}")
            raise
vectorstore=VectorStore()
vectorstore

In [ ]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(
    chunks,
    embeddings
)